# ORB Ensemble vs Buy and Hold (Client Friendly, Parametric)

This notebook is designed to be clear in one pass:
- full baseline and execution settings are explicit,
- sub-signals are built as full cross-product Q_LOW_VALUES x Q_HIGH_VALUES,
- only Ensemble vs Buy and Hold are plotted (dark Plotly style).


In [47]:
import math
import sys
from pathlib import Path

# Make `src` imports work whether notebook is launched from repo root or /notebooks
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from src.data.loader import load_ohlcv_file
from src.data.cleaning import clean_ohlcv
from src.features.intraday import add_intraday_features, add_session_vwap, add_continuous_session_vwap
from src.features.opening_range import compute_opening_range
from src.features.volatility import add_atr
from src.strategy.orb import ORBStrategy
from src.engine.execution_model import ExecutionModel
from src.engine.backtester import run_backtest
from src.engine.portfolio import build_equity_curve
from src.analytics.metrics import compute_metrics
from src.analytics.orb_notebook_utils import (
    build_scope_readout_markdown,
    curve_annualized_return,
    curve_daily_sharpe,
    curve_daily_vol,
    curve_drawdown_pct,
    curve_max_drawdown_pct,
    curve_total_return_pct,
    format_curve_stats_line,
    normalize_curve,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

## 1) Parameters (edit here)

In [48]:
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

# ----------------------
# Data / split settings
# ----------------------
DATASET_PATH = ROOT / 'data' / 'processed' / 'parquet' / 'MNQ_c_0_1m_20260321_094501.parquet'
IS_FRACTION = 0.70

# ----------------------
# Ensemble settings
# ----------------------
#OR15LONG
# ATR_PERIODS = list(range(30, 60 + 1, 1))
# Q_LOW_VALUES = list(range(25, 30 + 1, 1))
# Q_HIGH_VALUES = list(range(70, 75 + 1, 1))

#OR30LONG
ATR_PERIODS = list(range(15, 20 + 1, 1))
Q_LOW_VALUES = list(range(25, 30 + 1, 1))
Q_HIGH_VALUES = list(range(70, 75 + 1, 1))

#OR30BOTH
# ATR_PERIODS = list(range(10, 30 + 1, 1))
# Q_LOW_VALUES = list(range(20, 25 + 1, 1))
# Q_HIGH_VALUES = list(range(85, 90 + 1, 1))


AGGREGATION_RULE = 'majority_50'           # majority_50 | consensus_75 | consensus_100 | custom
CUSTOM_THRESHOLD = 0.90               # used only if AGGREGATION_RULE == 'custom'

# ----------------------
# Baseline strategy settings
# ----------------------
BASELINE = {
    'or_minutes': 30,
    'opening_time': '09:30:00',
    'direction': 'long',
    'one_trade_per_day': True,
    'entry_buffer_ticks': 2,
    'stop_buffer_ticks': 2,
    'target_multiple': 2.0,
    'vwap_confirmation': True,
    'vwap_column': 'continuous_session_vwap',
    'time_exit': '16:00:00',
    'account_size_usd': 50000.0,
    'risk_per_trade_pct': 1.5,
    'tick_size': 0.25,
    'entry_on_next_open': True,
}

# ----------------------
# Execution / costs
# ----------------------
EXECUTION = {
    'commission_per_side_usd': 0.62,
    'slippage_ticks': 1,
    'tick_size': 0.25,
}

print('DATASET_PATH   =', DATASET_PATH)
print('IS_FRACTION    =', IS_FRACTION)
print('ATR_PERIODS    =', ATR_PERIODS)
print('Q_LOW_VALUES   =', Q_LOW_VALUES)
print('Q_HIGH_VALUES  =', Q_HIGH_VALUES)
print('N_SUBSIGNALS_MAX =', len(ATR_PERIODS) * len(Q_LOW_VALUES) * len(Q_HIGH_VALUES))
print('AGG_RULE       =', AGGREGATION_RULE)

DATASET_PATH   = d:\Business\Trading\VSCODE\algo-trading-intraday-research\data\processed\parquet\MNQ_c_0_1m_20260321_094501.parquet
IS_FRACTION    = 0.7
ATR_PERIODS    = [15, 16, 17, 18, 19, 20]
Q_LOW_VALUES   = [25, 26, 27, 28, 29, 30]
Q_HIGH_VALUES  = [70, 71, 72, 73, 74, 75]
N_SUBSIGNALS_MAX = 216
AGG_RULE       = majority_50


## 2) Build data, baseline signals, and sub-signals

In [49]:
if AGGREGATION_RULE == 'majority_50':
    consensus_threshold = 0.50
elif AGGREGATION_RULE == 'consensus_75':
    consensus_threshold = 0.75
elif AGGREGATION_RULE == 'consensus_100':
    consensus_threshold = 1.00
elif AGGREGATION_RULE == 'custom':
    consensus_threshold = float(CUSTOM_THRESHOLD)
else:
    raise ValueError(f'Unknown AGGREGATION_RULE: {AGGREGATION_RULE}')

# 1) Feature set + ATR columns
raw = load_ohlcv_file(Path(DATASET_PATH))
raw = clean_ohlcv(raw)
feat = add_intraday_features(raw)
feat = add_session_vwap(feat)
feat = add_continuous_session_vwap(feat, session_start_hour=18)
feat = compute_opening_range(feat, or_minutes=int(BASELINE['or_minutes']), opening_time=str(BASELINE['opening_time']))
feat = add_atr(feat, window=ATR_PERIODS)
atr_cols = [f'atr_{p}' for p in ATR_PERIODS if f'atr_{p}' in feat.columns]
if not atr_cols:
    raise RuntimeError('No ATR columns generated. Check ATR_PERIODS.')

# 2) Baseline signals
strategy = ORBStrategy(
    or_minutes=int(BASELINE['or_minutes']),
    direction=str(BASELINE['direction']),
    one_trade_per_day=bool(BASELINE['one_trade_per_day']),
    entry_buffer_ticks=int(BASELINE['entry_buffer_ticks']),
    stop_buffer_ticks=int(BASELINE['stop_buffer_ticks']),
    target_multiple=float(BASELINE['target_multiple']),
    opening_time=str(BASELINE['opening_time']),
    time_exit=str(BASELINE['time_exit']),
    account_size_usd=float(BASELINE['account_size_usd']),
    risk_per_trade_pct=float(BASELINE['risk_per_trade_pct']),
    tick_size=float(BASELINE['tick_size']),
    vwap_confirmation=bool(BASELINE['vwap_confirmation']),
    vwap_column=str(BASELINE['vwap_column']),
)
signal_df = strategy.generate_signals(feat)

all_sessions = sorted(pd.to_datetime(signal_df['session_date']).dt.date.unique())
split_idx = int(len(all_sessions) * float(IS_FRACTION))
split_idx = max(1, min(len(all_sessions) - 1, split_idx))
is_sessions = all_sessions[:split_idx]
oos_sessions = all_sessions[split_idx:]

# Candidate days: first signal of each session
candidate_cols = ['session_date', 'timestamp', 'signal'] + atr_cols
candidates = signal_df.loc[signal_df['signal'].ne(0), candidate_cols].copy()
candidates = candidates.sort_values('timestamp').drop_duplicates(subset=['session_date'], keep='first')
if candidates.empty:
    raise RuntimeError('No candidate signal day found.')
candidates['signal_index'] = candidates.index

is_mask = candidates['session_date'].isin(set(is_sessions))
if not bool(is_mask.any()):
    raise RuntimeError('No IS candidate days available for quantile calibration.')

# 3) Backtest baseline once; each ATR cell reuses session filtering
exec_model = ExecutionModel(
    commission_per_side_usd=float(EXECUTION['commission_per_side_usd']),
    slippage_ticks=int(EXECUTION['slippage_ticks']),
    tick_size=float(EXECUTION['tick_size']),
)
baseline_trades = run_backtest(
    signal_df,
    execution_model=exec_model,
    time_exit=str(BASELINE['time_exit']),
    stop_buffer_ticks=int(BASELINE['stop_buffer_ticks']),
    target_multiple=float(BASELINE['target_multiple']),
    account_size_usd=float(BASELINE['account_size_usd']),
    risk_per_trade_pct=float(BASELINE['risk_per_trade_pct']),
    entry_on_next_open=bool(BASELINE['entry_on_next_open']),
)

pass_count = np.zeros(len(candidates), dtype=np.int32)
rows = []

for period in ATR_PERIODS:
    atr_col = f'atr_{period}'
    atr_all = pd.to_numeric(candidates[atr_col], errors='coerce')
    atr_is_values = atr_all.loc[is_mask].dropna()
    if atr_is_values.empty:
        continue

    for ql in Q_LOW_VALUES:
        for qh in Q_HIGH_VALUES:
            if ql >= qh:
                continue

            low_thr = float(atr_is_values.quantile(ql / 100.0))
            high_thr = float(atr_is_values.quantile(qh / 100.0))
            if (not np.isfinite(low_thr)) or (not np.isfinite(high_thr)) or low_thr >= high_thr:
                continue

            pass_mask = atr_all.between(low_thr, high_thr, inclusive='both').fillna(False)
            pass_count += pass_mask.to_numpy(dtype=np.int32)

            selected_rows = candidates.loc[pass_mask, ['session_date', 'signal_index']].copy()
            selected_sessions = set(selected_rows['session_date'])
            model_trades = baseline_trades.loc[baseline_trades['session_date'].isin(selected_sessions)].copy()

            is_sel = sorted(set(is_sessions).intersection(selected_sessions))
            oos_sel = sorted(set(oos_sessions).intersection(selected_sessions))
            m_is = compute_metrics(
                model_trades.loc[model_trades['session_date'].isin(set(is_sel))].copy(),
                session_dates=is_sel,
                initial_capital=float(BASELINE['account_size_usd']),
            )
            m_oos_active = compute_metrics(
                model_trades.loc[model_trades['session_date'].isin(set(oos_sel))].copy(),
                session_dates=oos_sel,
                initial_capital=float(BASELINE['account_size_usd']),
            )
            m_oos = compute_metrics(
                model_trades.loc[model_trades['session_date'].isin(set(oos_sessions))].copy(),
                session_dates=oos_sessions,
                initial_capital=float(BASELINE['account_size_usd']),
            )

            rows.append(
                {
                    'model_id': f'atr{period}_q{ql}_q{qh}',
                    'atr_period': int(period),
                    'q_low_pct': int(ql),
                    'q_high_pct': int(qh),
                    'low_threshold_is': float(low_thr),
                    'high_threshold_is': float(high_thr),
                    'n_selected_signals': int(len(selected_rows)),
                    'is_sharpe_ratio': float(m_is.get('sharpe_ratio', 0.0)),
                    'is_profit_factor': float(m_is.get('profit_factor', 0.0)),
                    'is_expectancy': float(m_is.get('expectancy', 0.0)),
                    'oos_selected_days': int(len(oos_sel)),
                    'oos_n_trades': int(m_oos.get('n_trades', 0)),
                    'oos_active_sharpe_ratio': float(m_oos_active.get('sharpe_ratio', 0.0)),
                    'oos_sharpe_ratio': float(m_oos.get('sharpe_ratio', 0.0)),
                    'oos_profit_factor': float(m_oos.get('profit_factor', 0.0)),
                    'oos_expectancy': float(m_oos.get('expectancy', 0.0)),
                }
            )

if not rows:
    raise RuntimeError('No valid sub-signals generated.')

submodels_df = pd.DataFrame(rows).sort_values(['atr_period', 'q_low_pct', 'q_high_pct']).reset_index(drop=True)
sweep_df = submodels_df.copy()

# Best cell for reporting (with trade floor)
trade_floor = 40
ranked = sweep_df.loc[sweep_df['oos_n_trades'] >= trade_floor].copy()
if ranked.empty:
    ranked = sweep_df.copy()
ranked = ranked.sort_values(
    ['oos_sharpe_ratio', 'oos_profit_factor', 'oos_expectancy'],
    ascending=[False, False, False],
).reset_index(drop=True)
best_row = ranked.iloc[0].to_dict()

# 4) Ensemble selection using all ATR/Q sub-signals
n_subsignals = int(len(submodels_df))
candidates['consensus_score'] = pass_count / float(max(n_subsignals, 1))
candidates['ensemble_pass'] = candidates['consensus_score'] >= consensus_threshold
selected_indices = pd.Index(candidates.loc[candidates['ensemble_pass'], 'signal_index'])

ensemble_signal_df = signal_df.copy()
sig_mask = ensemble_signal_df['signal'].ne(0)
ensemble_signal_df.loc[sig_mask & (~ensemble_signal_df.index.isin(set(selected_indices))), 'signal'] = 0

print('all_sessions      =', len(all_sessions))
print('is_sessions       =', len(is_sessions))
print('oos_sessions      =', len(oos_sessions))
print('candidate_days    =', len(candidates))
print('sub_signals_used  =', n_subsignals)
print('consensus_thr     =', consensus_threshold)
print('selected_days     =', int(candidates['ensemble_pass'].sum()))
print('best_cell         =', f"ATR {int(best_row['atr_period'])}, q{int(best_row['q_low_pct'])}/q{int(best_row['q_high_pct'])}")

all_sessions      = 2143
is_sessions       = 1500
oos_sessions      = 643
candidate_days    = 1244
sub_signals_used  = 216
consensus_thr     = 0.5
selected_days     = 532
best_cell         = ATR 16, q30/q73


## 3) Backtest ensemble + Buy and Hold benchmark

In [50]:
def _to_naive_utc(series: pd.Series) -> pd.Series:
    ts = pd.to_datetime(series, utc=True, errors='coerce')
    return ts.dt.tz_convert(None)


initial_capital = float(BASELINE['account_size_usd'])


def drawdown_pct_from_equity(eq: pd.Series) -> pd.Series:
    return curve_drawdown_pct(eq)


def sharpe_daily_from_equity(df: pd.DataFrame) -> float:
    return curve_daily_sharpe(df)


def annualized_return_from_equity(df: pd.DataFrame) -> float:
    return curve_annualized_return(df, initial_capital)


def vol_daily_from_equity(df: pd.DataFrame) -> float:
    return curve_daily_vol(df)


def total_return_pct(df: pd.DataFrame) -> float:
    return curve_total_return_pct(df, initial_capital)


def max_drawdown_pct(df: pd.DataFrame) -> float:
    return curve_max_drawdown_pct(df)


def format_stats_line(name: str, sharpe: float, ret_pct: float, cagr_pct: float, vol_pct: float, dd_pct: float, pf: float | None = None, exp: float | None = None) -> str:
    return format_curve_stats_line(
        name=name,
        sharpe=sharpe,
        ret_pct=ret_pct,
        cagr_pct=cagr_pct,
        vol_pct=vol_pct,
        dd_pct=dd_pct,
        pf=pf,
        exp=exp,
    )


# Run ensemble backtest with explicit execution settings
exec_model = ExecutionModel(
    commission_per_side_usd=float(EXECUTION['commission_per_side_usd']),
    slippage_ticks=int(EXECUTION['slippage_ticks']),
    tick_size=float(EXECUTION['tick_size']),
)

ensemble_trades = run_backtest(
    ensemble_signal_df,
    execution_model=exec_model,
    time_exit=str(BASELINE['time_exit']),
    stop_buffer_ticks=int(BASELINE['stop_buffer_ticks']),
    target_multiple=float(BASELINE['target_multiple']),
    account_size_usd=float(BASELINE['account_size_usd']),
    risk_per_trade_pct=float(BASELINE['risk_per_trade_pct']),
    entry_on_next_open=bool(BASELINE['entry_on_next_open']),
)

ensemble_eq = normalize_curve(
    build_equity_curve(ensemble_trades, initial_capital=float(BASELINE['account_size_usd']))
)

# Buy and hold from daily close
bench_src = feat[['timestamp', 'close', 'session_date']].copy()
bench_src['timestamp'] = _to_naive_utc(bench_src['timestamp'])
bench_src['close'] = pd.to_numeric(bench_src['close'], errors='coerce')
bench_src = bench_src.dropna(subset=['timestamp', 'close'])
daily_close = bench_src.groupby('session_date', as_index=True)['close'].last().dropna()

bench = normalize_curve(pd.DataFrame({
    'timestamp': pd.to_datetime(daily_close.index),
    'equity': float(BASELINE['account_size_usd']) * (daily_close / daily_close.iloc[0]),
}).sort_values('timestamp').reset_index(drop=True))

# Ensemble metrics
ensemble_overall = compute_metrics(
    ensemble_trades,
    signal_df=ensemble_signal_df,
    session_dates=all_sessions,
    initial_capital=float(BASELINE['account_size_usd']),
)
ensemble_is = compute_metrics(
    ensemble_trades.loc[ensemble_trades['session_date'].isin(set(is_sessions))].copy(),
    session_dates=is_sessions,
    initial_capital=float(BASELINE['account_size_usd']),
)
ensemble_oos = compute_metrics(
    ensemble_trades.loc[ensemble_trades['session_date'].isin(set(oos_sessions))].copy(),
    session_dates=oos_sessions,
    initial_capital=float(BASELINE['account_size_usd']),
)

print('ensemble_trades =', len(ensemble_trades))
print('ensemble_is_sharpe =', float(ensemble_is.get('sharpe_ratio', 0.0)))
print('ensemble_oos_sharpe =', float(ensemble_oos.get('sharpe_ratio', 0.0)))
print('ensemble_overall_sharpe =', float(ensemble_overall.get('sharpe_ratio', 0.0)))
print('buy_hold_total_return_pct =', total_return_pct(bench))

ensemble_trades = 532
ensemble_is_sharpe = 1.0806648722081282
ensemble_oos_sharpe = 1.2578767230010726
ensemble_overall_sharpe = 1.124512363708086
buy_hold_total_return_pct = 216.0265590417914


## 4) Dark Plotly chart (Ensemble vs Buy and Hold)

In [51]:
heat_src = sweep_df.copy()
heat_src['pair'] = 'q' + heat_src['q_low_pct'].astype(int).astype(str) + '/q' + heat_src['q_high_pct'].astype(int).astype(str)

heat_sh = heat_src.pivot_table(index='atr_period', columns='pair', values='oos_sharpe_ratio', aggfunc='mean').sort_index()
heat_pf = heat_src.pivot_table(index='atr_period', columns='pair', values='oos_profit_factor', aggfunc='mean').sort_index()

fig_heat = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.08,
    subplot_titles=('OOS Sharpe (full OOS calendar)', 'OOS Profit Factor (full OOS calendar)')
)
fig_heat.add_trace(
    go.Heatmap(z=heat_sh.to_numpy(), x=heat_sh.columns.tolist(), y=heat_sh.index.tolist(), colorscale='RdYlGn', colorbar=dict(title='Sharpe', x=0.46), zmin=0.5, zmax=1.5),
    row=1,
    col=1,
)
fig_heat.add_trace(
    go.Heatmap(z=heat_pf.to_numpy(), x=heat_pf.columns.tolist(), y=heat_pf.index.tolist(), colorscale='RdYlGn', colorbar=dict(title='PF', x=1.01), zmin=0.5, zmax=1.5),
    row=1,
    col=2,
)
fig_heat.update_layout(title='Parameter Heatmaps', width=1850, height=650, template='plotly_dark')
fig_heat.update_xaxes(title_text='Quantile pair', tickangle=45)
fig_heat.update_yaxes(title_text='ATR period')
fig_heat.show()

trend_df = (
    sweep_df.groupby('atr_period', as_index=False)
    .agg(
        median_oos_sharpe=('oos_sharpe_ratio', 'median'),
        median_oos_pf=('oos_profit_factor', 'median'),
        median_oos_expectancy=('oos_expectancy', 'median'),
        median_oos_selected_days=('oos_selected_days', 'median'),
    )
    .sort_values('atr_period')
)

fig_trend = make_subplots(rows=1, cols=3, subplot_titles=('Median OOS Sharpe', 'Median OOS PF', 'Median OOS Expectancy'))
fig_trend.add_trace(go.Scatter(x=trend_df['atr_period'], y=trend_df['median_oos_sharpe'], mode='lines+markers', line=dict(color='#22c55e')), row=1, col=1)
fig_trend.add_trace(go.Scatter(x=trend_df['atr_period'], y=trend_df['median_oos_pf'], mode='lines+markers', line=dict(color='#38bdf8')), row=1, col=2)
fig_trend.add_trace(go.Scatter(x=trend_df['atr_period'], y=trend_df['median_oos_expectancy'], mode='lines+markers', line=dict(color='#f59e0b')), row=1, col=3)
fig_trend.update_layout(template='plotly_dark', width=1850, height=520, title='Metrics Evolution by ATR')
fig_trend.update_xaxes(title_text='ATR period')
fig_trend.show()

best_atr_row = trend_df.sort_values(['median_oos_sharpe', 'median_oos_pf'], ascending=[False, False]).iloc[0]
display(Markdown(
    '### Quick summary\n'
    f"- Best single cell (full OOS): **ATR {int(best_row['atr_period'])} | q{int(best_row['q_low_pct'])}/q{int(best_row['q_high_pct'])}**\n"
    f"- Best ATR by median full-OOS behavior: **ATR {int(best_atr_row['atr_period'])}**\n"
    f"- Median selected OOS days at best ATR: **{best_atr_row['median_oos_selected_days']:.0f}** / **{len(oos_sessions)}**\n"
    f"- Trade floor used for ranking: **{trade_floor}**"
))

# -----------------------------
# Ensemble vs Buy and Hold plot
# -----------------------------
ens_ret = total_return_pct(ensemble_eq)
ens_dd = max_drawdown_pct(ensemble_eq)
ens_cagr = annualized_return_from_equity(ensemble_eq) * 100.0
ens_vol = vol_daily_from_equity(ensemble_eq) * 100.0
ens_overall_sh = float(ensemble_overall.get('sharpe_ratio', 0.0))
ens_overall_pf = float(ensemble_overall.get('profit_factor', 0.0))
ens_overall_exp = float(ensemble_overall.get('expectancy', 0.0))
ens_is_sh = float(ensemble_is.get('sharpe_ratio', 0.0))
ens_oos_sh = float(ensemble_oos.get('sharpe_ratio', 0.0))
ens_oos_pf = float(ensemble_oos.get('profit_factor', 0.0))
ens_oos_exp = float(ensemble_oos.get('expectancy', 0.0))

bench_ret = total_return_pct(bench)
bench_dd = max_drawdown_pct(bench)
bench_sh = sharpe_daily_from_equity(bench)
bench_cagr = annualized_return_from_equity(bench) * 100.0
bench_vol = vol_daily_from_equity(bench) * 100.0
bench_is = bench.loc[bench['timestamp'].dt.date.isin(set(is_sessions))].copy()
bench_oos = bench.loc[bench['timestamp'].dt.date.isin(set(oos_sessions))].copy()
bench_is_sh = sharpe_daily_from_equity(bench_is)
bench_oos_sh = sharpe_daily_from_equity(bench_oos)

legend_ens = (
    f"Ensemble | Overall Sharpe {ens_overall_sh:.2f} | PF {ens_overall_pf:.2f} | "
    f"Exp {ens_overall_exp:.1f} | DD {float(ensemble_overall.get('max_drawdown', 0.0)):.0f}"
)
legend_bh = (
    f"Buy&Hold | Overall Sharpe {bench_sh:.2f} | Ret {bench_ret:.1f}% | "
    f"CAGR {bench_cagr:.1f}% | Vol {bench_vol:.1f}% | MaxDD {bench_dd:.1f}%"
)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.075,
    row_heights=[0.70, 0.30],
    subplot_titles=('Equity Curve (USD)', 'Drawdown (%)')
)

fig.add_trace(go.Scatter(x=ensemble_eq['timestamp'], y=ensemble_eq['equity'], mode='lines', name=legend_ens, line=dict(width=3.0, color='#22c55e')), row=1, col=1)
fig.add_trace(go.Scatter(x=ensemble_eq['timestamp'], y=ensemble_eq['drawdown_pct'], mode='lines', name='DD Ensemble', showlegend=False, line=dict(width=1.7, color='#22c55e', dash='dot')), row=2, col=1)
fig.add_trace(go.Scatter(x=bench['timestamp'], y=bench['equity'], mode='lines', name=legend_bh, line=dict(width=2.6, color='#38bdf8')), row=1, col=1)
fig.add_trace(go.Scatter(x=bench['timestamp'], y=bench['drawdown_pct'], mode='lines', name='DD Buy&Hold', showlegend=False, line=dict(width=1.5, color='#38bdf8', dash='dot')), row=2, col=1)

summary_lines = [
    format_stats_line('Ensemble (overall)', ens_overall_sh, ens_ret, ens_cagr, ens_vol, ens_dd, pf=ens_overall_pf, exp=ens_overall_exp),
    f"<b>Ensemble split</b> | IS Sharpe {ens_is_sh:.2f} | OOS Sharpe {ens_oos_sh:.2f} | OOS PF {ens_oos_pf:.2f} | OOS Exp {ens_oos_exp:.1f}",
    f"<b>Buy&Hold</b> | Overall Sharpe {bench_sh:.2f} | IS Sharpe {bench_is_sh:.2f} | OOS Sharpe {bench_oos_sh:.2f} | Ret {bench_ret:.1f}% | CAGR {bench_cagr:.1f}% | Vol {bench_vol:.1f}% | MaxDD {bench_dd:.1f}%",
    f"<b>Params</b> | ATR {ATR_PERIODS} | Qlow {Q_LOW_VALUES} | Qhigh {Q_HIGH_VALUES} | agg {AGGREGATION_RULE} | thr {consensus_threshold:.2f} | Nsub {n_subsignals}",
    f"<b>Best cell</b> | ATR {int(best_row['atr_period'])} | q{int(best_row['q_low_pct'])}/q{int(best_row['q_high_pct'])} | OOS days {int(best_row['oos_selected_days'])}/{len(oos_sessions)}",
]

fig.add_annotation(
    xref='paper', yref='paper', x=0.01, y=1.22,
    text='<br>'.join(summary_lines),
    showarrow=False, align='left',
    bordercolor='rgba(148,163,184,0.40)', borderwidth=1, borderpad=10,
    bgcolor='rgba(15,23,42,0.94)', font=dict(size=13, color='#e5e7eb'),
)

fig.update_layout(
    template='plotly_dark',
    width=1850,
    height=1020,
    title=dict(text='Ensemble vs Buy&Hold', x=0.5, xanchor='center', font=dict(size=24)),
    legend=dict(orientation='h', yanchor='bottom', y=-0.24, xanchor='left', x=0.0, font=dict(size=11), bgcolor='rgba(15,23,42,0.82)', bordercolor='rgba(148,163,184,0.25)', borderwidth=1),
    margin=dict(l=70, r=40, t=195, b=140),
    paper_bgcolor='#020617',
    plot_bgcolor='#020617',
)
fig.update_annotations(font=dict(size=16, color='#e5e7eb'))
fig.update_yaxes(title_text='Equity (USD)', row=1, col=1)
fig.update_yaxes(title_text='Drawdown %', row=2, col=1)
fig.update_xaxes(title_text='Time', row=2, col=1)

fig.show()

### Quick summary
- Best single cell (full OOS): **ATR 16 | q30/q73**
- Best ATR by median full-OOS behavior: **ATR 18**
- Median selected OOS days at best ATR: **132** / **643**
- Trade floor used for ranking: **40**

## 5) KPI table

In [52]:
kpi = pd.DataFrame([
    {
        'model': 'ensemble',
        'overall_sharpe': ens_overall_sh,
        'is_sharpe': ens_is_sh,
        'oos_sharpe': ens_oos_sh,
        'overall_profit_factor': ens_overall_pf,
        'oos_profit_factor': ens_oos_pf,
        'overall_expectancy': ens_overall_exp,
        'oos_expectancy': ens_oos_exp,
        'total_return_pct': ens_ret,
        'cagr_pct': ens_cagr,
        'vol_pct': ens_vol,
        'max_drawdown_pct': ens_dd,
        'n_subsignals': n_subsignals,
        'selected_signal_days': int(candidates['ensemble_pass'].sum()),
    },
    {
        'model': 'buy_and_hold',
        'overall_sharpe': bench_sh,
        'is_sharpe': bench_is_sh,
        'oos_sharpe': bench_oos_sh,
        'overall_profit_factor': np.nan,
        'oos_profit_factor': np.nan,
        'overall_expectancy': np.nan,
        'oos_expectancy': np.nan,
        'total_return_pct': bench_ret,
        'cagr_pct': bench_cagr,
        'vol_pct': bench_vol,
        'max_drawdown_pct': bench_dd,
        'n_subsignals': np.nan,
        'selected_signal_days': np.nan,
    },
])
display(kpi)

display(Markdown(
    '### Top parameter cells (full-calendar OOS)\n'
    'Heatmaps and ranking below use the entire OOS window so they stay comparable to the ensemble OOS metrics.'
))
display(
    ranked[
        [
            'atr_period',
            'q_low_pct',
            'q_high_pct',
            'oos_selected_days',
            'oos_n_trades',
            'oos_sharpe_ratio',
            'oos_profit_factor',
            'oos_expectancy',
        ]
    ].head(12)
)

,model,overall_sharpe,is_sharpe,oos_sharpe,overall_profit_factor,oos_profit_factor,overall_expectancy,oos_expectancy,total_return_pct,cagr_pct,vol_pct,max_drawdown_pct,n_subsignals,selected_signal_days
0,ensemble,1.124512,1.080665,1.257877,1.410261,1.560617,95.985301,114.550303,102.128360,11.042477,16.102280,-8.962340,216.0,532.0
1,buy_and_hold,0.722141,0.738497,0.691994,NaN,NaN,NaN,NaN,216.026559,18.219680,22.143272,-35.314391,NaN,NaN


### Top parameter cells (full-calendar OOS)
Heatmaps and ranking below use the entire OOS window so they stay comparable to the ensemble OOS metrics.

,atr_period,q_low_pct,q_high_pct,oos_selected_days,oos_n_trades,oos_sharpe_ratio,oos_profit_factor,oos_expectancy
0,16,30,73,129,129,1.493467,1.713046,138.360310
1,16,30,74,133,133,1.479780,1.685350,136.179699
2,15,30,74,134,134,1.424870,1.647372,130.714328
3,17,30,72,120,120,1.392298,1.681400,134.124333
4,20,30,74,138,138,1.386759,1.613149,123.690145
5,18,29,72,127,127,1.377800,1.650632,128.793228
6,18,30,72,126,126,1.372123,1.647945,129.279206
7,16,30,72,124,124,1.370507,1.654825,128.538871
8,17,30,75,139,139,1.369911,1.599438,123.953525
9,16,29,73,131,131,1.368470,1.625345,125.937557
